# Desarrollo G1 - Practica ETL Semana 1

Dominio de negocio: ciberseguridad y monitoreo de incidentes de red.

Fuente de datos: Kaggle, dataset `teamincribo/cyber-security-attacks`.

Objetivo: configurar PostgreSQL en Docker, cargar una tabla de hechos de alertas de red y analizar tres fuentes relacionadas: PostgreSQL, CSV y JSON.

## 1. Configuracion inicial

Las credenciales de PostgreSQL se leen desde `.env`. Esto evita escribir usuarios, claves o puertos directamente en las celdas del notebook.

In [1]:
from pathlib import Path
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()
RAW_DIR = Path('data/raw')
postgres_url = (
    f"postgresql+psycopg2://{os.environ['POSTGRES_USER']}:{os.environ['POSTGRES_PASSWORD']}"
    f"@{os.environ.get('POSTGRES_HOST', 'localhost')}:{os.environ.get('POSTGRES_PORT', '5432')}/{os.environ['POSTGRES_DB']}"
)
engine = create_engine(postgres_url)
sorted(p.name for p in RAW_DIR.iterdir())

['asset_inventory.csv',
 'cybersecurity_attacks.csv',
 'vulnerability_catalog.json']

## 2. DataFrame 1 - PostgreSQL: network_alerts

Este DataFrame proviene de la base PostgreSQL levantada con Docker. La tabla `network_alerts` funciona como tabla de hechos del proyecto SOC, con eventos, protocolos, severidad, tamano de paquete y puntajes de anomalia.

In [2]:
df_alerts_pg = pd.read_sql('SELECT * FROM network_alerts', engine)
df_alerts_pg.head(5)

,timestamp,source_ip_address,destination_ip_address,source_port,destination_port,protocol,packet_length,packet_type,traffic_type,payload_data,...,user_information,device_information,network_segment,geo_location_data,proxy_information,firewall_logs,ids_ips_alerts,log_source,machine_id,alert_id
0,2023-05-30 06:33:58,103.216.15.12,84.9.164.252,31225,17616,ICMP,503,Data,HTTP,Qui natus odio asperiores nam. Optio nobis ius...,...,Reyansh Dugal,Mozilla/5.0 (compatible; MSIE 8.0; Windows NT ...,Segment A,"Jamshedpur, Sikkim",150.9.97.135,Log Data,NaN,Server,SRV-0001,MALWARE-ICMP
1,2020-08-26 07:08:30,78.199.217.198,66.191.137.154,17245,48166,ICMP,1174,Data,HTTP,Aperiam quos modi officiis veritatis rem. Omni...,...,Sumer Rana,Mozilla/5.0 (compatible; MSIE 8.0; Windows NT ...,Segment B,"Bilaspur, Nagaland",NaN,Log Data,NaN,Firewall,SRV-0002,MALWARE-ICMP
2,2022-11-13 08:23:25,63.79.210.48,198.219.82.17,16811,53600,UDP,306,Control,HTTP,Perferendis sapiente vitae soluta. Hic delectu...,...,Himmat Karpe,Mozilla/5.0 (compatible; MSIE 9.0; Windows NT ...,Segment C,"Bokaro, Rajasthan",114.133.48.179,Log Data,Alert Data,Firewall,SRV-0003,DDOS-UDP
3,2023-07-02 10:38:46,163.42.196.10,101.228.192.255,20018,32534,UDP,385,Data,HTTP,Totam maxime beatae expedita explicabo porro l...,...,Fateh Kibe,Mozilla/5.0 (Macintosh; PPC Mac OS X 10_11_5; ...,Segment B,"Jaunpur, Rajasthan",NaN,NaN,Alert Data,Firewall,SRV-0004,MALWARE-UDP
4,2023-07-16 13:11:07,71.166.185.76,189.243.174.238,6131,26646,TCP,1462,Data,DNS,Odit nesciunt dolorem nisi iste iusto. Animi v...,...,Dhanush Chad,Mozilla/5.0 (compatible; MSIE 5.0; Windows NT ...,Segment C,"Anantapur, Tripura",149.6.110.119,NaN,Alert Data,Firewall,SRV-0005,DDOS-TCP


In [3]:
pd.DataFrame({
    'columna': df_alerts_pg.columns,
    'tiene_nulos': df_alerts_pg.isna().any().values,
    'valores_faltantes': df_alerts_pg.isna().sum().values,
})

,columna,tiene_nulos,valores_faltantes
0,timestamp,False,0
1,source_ip_address,False,0
2,destination_ip_address,False,0
3,source_port,False,0
4,destination_port,False,0
5,protocol,False,0
6,packet_length,False,0
7,packet_type,False,0
8,traffic_type,False,0
9,payload_data,False,0


In [4]:
df_alerts_pg[['packet_length', 'anomaly_scores']].agg(['mean', 'max', 'min']).round(2)

,packet_length,anomaly_scores
mean,788.3,50.13
max,1500.0,99.99
min,64.0,0.00


In [5]:
(
    df_alerts_pg.groupby(['protocol', 'severity_level'])[['packet_length', 'anomaly_scores']]
    .agg(['max', 'min']).head(12).round(2)
)

packet_length     anomaly_scores      
                                  max min            max   min
protocol severity_level                                       
ICMP     High                    1500  65          99.92  0.06
         Low                     1498  64          99.65  0.17
         Medium                  1495  64          99.93  0.06
TCP      High                    1499  66          99.99  0.01
         Low                     1500  64          99.93  0.12
         Medium                  1500  64          99.81  0.08
UDP      High                    1499  64          99.98  0.02
         Low                     1499  64          99.82  0.31
         Medium                  1500  65          99.97  0.00

## 3. DataFrame 2 - CSV: asset_inventory

Este DataFrame representa el inventario de activos de infraestructura. Se relaciona con `network_alerts` mediante `machine_id`, creado a partir de la IP destino de las alertas.

In [6]:
df_assets_csv = pd.read_csv(RAW_DIR / 'asset_inventory.csv')
df_assets_csv.head(5)

,machine_id,server_name,operating_system,department,ip_address,network_segment,criticality,log_source
0,SRV-0001,soc-node-0001,Windows,Tecnologia,84.9.164.252,Segment A,Low,Server
1,SRV-0002,soc-node-0002,Windows,Finanzas,66.191.137.154,Segment B,Low,Firewall
2,SRV-0003,soc-node-0003,Windows,Operaciones,198.219.82.17,Segment C,Low,Firewall
3,SRV-0004,soc-node-0004,Mac OS X,Finanzas,101.228.192.255,Segment B,Medium,Firewall
4,SRV-0005,soc-node-0005,Windows,Operaciones,189.243.174.238,Segment C,Low,Firewall


In [7]:
pd.DataFrame({
    'columna': df_assets_csv.columns,
    'tiene_nulos': df_assets_csv.isna().any().values,
    'valores_faltantes': df_assets_csv.isna().sum().values,
})

,columna,tiene_nulos,valores_faltantes
0,machine_id,False,0
1,server_name,False,0
2,operating_system,False,0
3,department,False,0
4,ip_address,False,0
5,network_segment,False,0
6,criticality,False,0
7,log_source,False,0


In [8]:
df_assets_csv['criticality_score'] = df_assets_csv['criticality'].map({'Low': 1, 'Medium': 2, 'High': 3}).fillna(0)
df_assets_csv[['criticality_score']].agg(['mean', 'max', 'min']).round(2)

,criticality_score
mean,1.9
max,3.0
min,1.0


In [9]:
(
    df_assets_csv.groupby(['department', 'operating_system'])[['criticality_score']]
    .agg(['max', 'min']).head(12).round(2)
)

criticality_score    
                                           max min
department  operating_system                      
Finanzas    Android                          1   1
            Linux                            3   1
            Mac OS X                         3   1
            Windows                          3   1
Operaciones Android                          3   3
            Linux                            3   1
            Mac OS X                         3   1
            Windows                          3   1
Tecnologia  Android                          3   1
            Linux                            3   1
            Mac OS X                         3   1
            Windows                          3   1

## 4. DataFrame 3 - JSON: vulnerability_catalog

Este DataFrame se lee desde `vulnerability_catalog.json`. El catalogo mapea tipos de ataque y protocolos con CVE simulados, puntaje CVSS y accion de remediacion. Se relaciona con las alertas mediante `attack_type`, `protocol` y `severity_level`.

In [10]:
df_vulns_json = pd.read_json(RAW_DIR / 'vulnerability_catalog.json')
df_vulns_json.head(5)

,alert_id,attack_type,protocol,severity_level,vulnerability_name,cve_code,cvss_score,remediation_action
0,ALRT-001,DDoS,ICMP,High,DDoS exposure over ICMP,CVE-2024-0001,9.23,"Revisar reglas IDS/IPS, endurecer ICMP y prior..."
1,ALRT-002,DDoS,ICMP,Low,DDoS exposure over ICMP,CVE-2024-0002,4.27,"Revisar reglas IDS/IPS, endurecer ICMP y prior..."
2,ALRT-003,DDoS,ICMP,Medium,DDoS exposure over ICMP,CVE-2024-0003,6.91,"Revisar reglas IDS/IPS, endurecer ICMP y prior..."
3,ALRT-004,DDoS,TCP,High,DDoS exposure over TCP,CVE-2024-0004,9.22,"Revisar reglas IDS/IPS, endurecer TCP y priori..."
4,ALRT-005,DDoS,TCP,Low,DDoS exposure over TCP,CVE-2024-0005,4.27,"Revisar reglas IDS/IPS, endurecer TCP y priori..."


In [11]:
pd.DataFrame({
    'columna': df_vulns_json.columns,
    'tiene_nulos': df_vulns_json.isna().any().values,
    'valores_faltantes': df_vulns_json.isna().sum().values,
})

,columna,tiene_nulos,valores_faltantes
0,alert_id,False,0
1,attack_type,False,0
2,protocol,False,0
3,severity_level,False,0
4,vulnerability_name,False,0
5,cve_code,False,0
6,cvss_score,False,0
7,remediation_action,False,0


In [12]:
df_vulns_json[['cvss_score']].agg(['mean', 'max', 'min']).round(2)

,cvss_score
mean,6.80
max,9.23
min,4.27


In [13]:
(
    df_vulns_json.groupby(['protocol', 'severity_level'])[['cvss_score']]
    .agg(['max', 'min']).head(12).round(2)
)

cvss_score      
                               max   min
protocol severity_level                 
ICMP     High                 9.23  9.19
         Low                  4.31  4.27
         Medium               6.91  6.89
TCP      High                 9.22  9.21
         Low                  4.32  4.27
         Medium               6.91  6.90
UDP      High                 9.21  9.19
         Low                  4.31  4.29
         Medium               6.91  6.90

## 5. Relacion entre fuentes

Se integran las tres fuentes para responder preguntas de inteligencia de negocio en un contexto SOC: que departamentos concentran mas alertas, que protocolos tienen mayor riesgo y que activos requieren priorizacion.

In [14]:
df_integrated = (
    df_alerts_pg.merge(df_assets_csv, on='machine_id', how='left')
    .merge(df_vulns_json, on=['attack_type', 'protocol', 'severity_level'], how='left')
)
(
    df_integrated.groupby(['department', 'severity_level'])[['anomaly_scores', 'cvss_score']]
    .agg(['mean', 'max', 'min', 'count']).round(2).head(12)
)

anomaly_scores                    cvss_score        \
                                     mean    max   min count       mean   max   
department  severity_level                                                      
Finanzas    High                    49.91  91.04  0.90    28       9.21  9.23   
            Low                     54.96  97.16  3.03    28       4.29  4.31   
            Medium                  42.07  93.93  2.26    27       6.90  6.91   
Operaciones High                    50.50  96.27  4.59    22       9.21  9.23   
            Low                     53.86  98.88  0.12    40       4.29  4.32   
            Medium                  48.82  97.64  2.88    24       6.90  6.91   
Tecnologia  High                    55.95  98.62  1.33    29       9.21  9.23   
            Low                     45.57  91.08  0.19    36       4.29  4.32   
            Medium                  57.41  96.27  1.31    16       6.90  6.91   

                                        
                             min count  
department  severity_level              
Finanzas    High            9.19    28  
            Low             4.27    28  
            Medium          6.89    27  
Operaciones High            9.19    22  
            Low             4.27    40  
            Medium          6.89    24  
Tecnologia  High            9.19    29  
            Low             4.27    36  
            Medium          6.90    16

# Aplicacion Profesional de la Practica

## Carlos Diaz

En mi contexto profesional, esta practica puede aplicarse directamente al area de infraestructura tecnologica, operaciones cloud y ciberseguridad. En este tipo de entorno se generan datos de logs de red, eventos de firewall, alertas IDS/IPS, inventario de servidores, sistemas operativos, direcciones IP, severidad de incidentes, acciones tomadas y vulnerabilidades asociadas. Normalmente estos datos se encuentran dispersos entre consolas, archivos CSV, servicios cloud, herramientas de monitoreo y reportes manuales.

PostgreSQL, Docker y Python permiten ordenar ese ecosistema. Docker facilita levantar una base de datos reproducible para pruebas o analisis sin depender de configuraciones manuales de una sola computadora. PostgreSQL permite almacenar de forma estructurada las alertas y consultarlas con criterios consistentes. Python ayuda a extraer archivos de Kaggle u otras fuentes, transformar columnas, detectar nulos, crear claves de relacion y generar indicadores de riesgo.

Un proceso ETL aportaria trazabilidad, repetibilidad y mejor toma de decisiones. En vez de revisar logs aislados, la organizacion podria integrar alertas, activos y vulnerabilidades en un modelo comun. Esto ayudaria a identificar departamentos con mayor exposicion, protocolos mas atacados, servidores criticos con alertas recurrentes y vulnerabilidades que requieren remediacion prioritaria.

Con esta informacion se podrian resolver problemas concretos: priorizar parches, reducir falsos positivos, justificar inversiones de seguridad, definir reglas de firewall, medir el comportamiento de incidentes por segmento de red y entregar reportes ejecutivos basados en evidencia. La practica demuestra como pasar de archivos sueltos a informacion confiable para operar mejor un entorno tecnologico.